# 2 - Voice cloning (Qwen3-TTS 1.7B **Base**)

Point at a wav of someone speaking, give the matching transcript, and it reads any text
in that voice. This is a different checkpoint from notebook 1: the Base model clones but
**cannot take a narration direction** - there is no instruct argument on this API.

### What testing showed

- **The reference transcript must match the audio exactly.** A truncated transcript
  changes the output.
- **The clone inherits delivery, not just timbre.** An 8 s clip of a lively passage gives
  a lively narrator. Record the register you actually want in the app.
- **Cloned output is slightly washy.** Dereverbing the reference was tried and rejected -
  it removed the reverb and the resemblance together.
- 15-20 s of reference is plenty. Level it; do not denoise it.
- Cross-language cloning (an English sample reading Spanish) is untested. There is a cell
  for it below.

In [1]:
# ============================ DEVICE ============================
# Choose the GPU BEFORE torch is imported.
#   "1"   -> RTX 4060   (safe: leaves the 4090 to ComfyUI)
#   "0"   -> RTX 4090   (fast, but only when ComfyUI is idle)
#   ""    -> CPU only   (very slow; works with no free GPU)
#
# The production voice worker also lives on GPU 1. While it runs there is only
# ~2.3 GB free there, which is NOT enough for the 1.7B model (~4.2 GB).
# Either stop it:  cd ~/ai_voice && docker compose down
# or set GPU = "0".
# Driver here is 615.71.09 / CUDA 13.4 -> install the cu130 wheels
# (see requirements.txt). Older cu12x wheels also run, because the
# driver is backward compatible.
import os
GPU = "1"

os.environ["CUDA_VISIBLE_DEVICES"] = GPU
DEVICE = "cuda:0" if GPU != "" else "cpu"   # after masking, the chosen card is always cuda:0
print("CUDA_VISIBLE_DEVICES =", GPU or "(none - CPU)")

CUDA_VISIBLE_DEVICES = 1


In [2]:
import time, json, subprocess
import numpy as np, torch, soundfile as sf
from IPython.display import Audio, display
from qwen_tts.inference.qwen3_tts_model import Qwen3TTSModel

MODELS = "/home/pavlo/lab/TTS/models"
OUT    = "/home/pavlo/lab/TTS/out"
os.makedirs(OUT, exist_ok=True)

def gpu_free():
    if DEVICE == "cpu": return "cpu"
    return subprocess.run(["nvidia-smi","--query-gpu=index,name,memory.free","--format=csv,noheader"],
                          capture_output=True, text=True).stdout.strip()
print(gpu_free())


********
********
 
0, NVIDIA GeForce RTX 4090, 5841 MiB
1, NVIDIA GeForce RTX 4060, 7885 MiB


In [3]:
# Loudness. Production target is -16 LUFS integrated, -1 dBTP ceiling, so narration
# sits predictably under the LTX ambient audio. Set NORMALISE = False for raw output.
NORMALISE   = True
TARGET_LUFS = -16.0

def normalise(a, sr):
    if not NORMALISE: return a
    try:
        import pyloudnorm as pyln
    except ImportError:
        print("pyloudnorm not installed - skipping"); return a
    m = pyln.Meter(sr)
    y = a * (10 ** ((TARGET_LUFS - m.integrated_loudness(a)) / 20.0))
    ceiling = 10 ** (-1.0/20.0)
    pk = float(np.max(np.abs(y)))
    if pk > ceiling: y = y * (ceiling / pk)
    return y.astype(np.float32)

def play(a, sr, name=None):
    a = normalise(np.asarray(a, dtype=np.float32), sr)
    if name:
        p = os.path.join(OUT, name if name.endswith(".wav") else name + ".wav")
        sf.write(p, a, sr, subtype="PCM_16"); print("saved", p)
    display(Audio(a, rate=sr))
    return a

## Load the cloning model

In [4]:
MODEL = MODELS + "/qwen3tts-17b-base"      # 1.7B Base - the cloning checkpoint

t0 = time.time()
model = Qwen3TTSModel.from_pretrained(
    MODEL, device_map=DEVICE,
    dtype=torch.bfloat16 if DEVICE != "cpu" else torch.float32)
print(f"loaded in {time.time()-t0:.0f}s")

loaded in 1s


## Prepare the reference clip

Drop a wav or m4a into voice_samples/ and point REF_IN at it. This converts to mono
24 kHz and levels to -20 LUFS. It deliberately does **not** denoise or dereverb.

REF_TEXT must match the audio word for word.

In [5]:
REF_IN  = "/home/pavlo/lab/TTS/voice_samples/pavlo_18s.wav"
REF_WAV = "/home/pavlo/lab/TTS/voice_samples/_ref_prepared.wav"

REF_TEXT = ("The lighthouse keeper woke before dawn, when the sky was still the colour of old "
            "silver. He filled his kettle, watched the steam curl against the cold window, and "
            "thought about the ships that had passed in the night. Down on the shore, at the "
            "edge of the water, a young fox was digging through the seaweed.")

subprocess.run(["ffmpeg","-y","-loglevel","error","-i",REF_IN,"-ac","1","-ar","24000",
                "/tmp/_ref_raw.wav"], check=True)
ref, rsr = sf.read("/tmp/_ref_raw.wav", dtype="float32")

try:
    import pyloudnorm as pyln
    lufs = pyln.Meter(rsr).integrated_loudness(ref)
    ref  = ref * (10 ** ((-20.0 - lufs) / 20.0))
    pk = float(np.max(np.abs(ref)))
    if pk > 0.95: ref = ref * (0.95 / pk)
    print(f"reference {len(ref)/rsr:.1f}s   {lufs:.1f} -> -20.0 LUFS   peak {float(np.max(np.abs(ref))):.2f}")
except ImportError:
    print("pyloudnorm missing - using the clip as-is")

sf.write(REF_WAV, ref.astype(np.float32), rsr, subtype="PCM_16")
display(Audio(ref, rate=rsr))

reference 18.9s   -20.0 -> -20.0 LUFS   peak 0.71


## Generate in that voice

x_vector_only_mode=True uses only the speaker embedding and ignores the reference audio,
which drops the room acoustics - but in testing it sounded **less** like the person.
Left as a switch; default off.

In [6]:
LANGUAGE      = "english"   # english / spanish / french / german / italian /
                           # portuguese / russian / japanese / korean / chinese
SEED          = 1234
X_VECTOR_ONLY = False      # True = timbre without room, weaker resemblance

TEXT = """Peppa smiled and gently reminded them that sharing is just as important
as learning. The friends nodded in agreement."""

if SEED is not None:
    torch.manual_seed(SEED); np.random.seed(SEED)

t0 = time.time()
wavs, sr = model.generate_voice_clone(
    text=" ".join(TEXT.split()),
    ref_audio=REF_WAV, ref_text=REF_TEXT,
    language=LANGUAGE, x_vector_only_mode=X_VECTOR_ONLY)
a = np.asarray(wavs[0], dtype=np.float32)
gen = time.time() - t0
print(f"{len(a)/sr:.2f}s audio in {gen:.2f}s (RTF {gen/(len(a)/sr):.3f})")
_ = play(a, sr, name="clone_" + LANGUAGE)

Setting `pad_token_id` to `eos_token_id`:2150 for open-end generation.


6.80s audio in 10.99s (RTF 1.616)
saved /home/pavlo/lab/TTS/out/clone_english.wav


## Same voice, several languages

Does an English reference read Spanish convincingly? Untested - this cell answers it.
If it does not convince, a parent can simply record a sample in that language instead.

In [7]:
SAMPLES = {
    "english": "Peppa smiled and gently reminded them that sharing is just as important as learning.",
    "spanish": "Peppa sonrio y les recordo con dulzura que compartir es tan importante como aprender.",
    "french":  "Peppa sourit et lui rappela doucement que partager est aussi important qu apprendre.",
    "german":  "Peppa laechelte und erinnerte sie sanft daran, dass Teilen genauso wichtig ist wie Lernen.",
}

for lang, txt in SAMPLES.items():
    if SEED is not None:
        torch.manual_seed(SEED); np.random.seed(SEED)
    wavs, sr = model.generate_voice_clone(text=txt, ref_audio=REF_WAV, ref_text=REF_TEXT,
                                          language=lang, x_vector_only_mode=X_VECTOR_ONLY)
    print("---", lang)
    play(np.asarray(wavs[0], dtype=np.float32), sr, name="clone_" + lang)

Setting `pad_token_id` to `eos_token_id`:2150 for open-end generation.


--- english
saved /home/pavlo/lab/TTS/out/clone_english.wav


Setting `pad_token_id` to `eos_token_id`:2150 for open-end generation.


--- spanish
saved /home/pavlo/lab/TTS/out/clone_spanish.wav


Setting `pad_token_id` to `eos_token_id`:2150 for open-end generation.


--- french
saved /home/pavlo/lab/TTS/out/clone_french.wav


Setting `pad_token_id` to `eos_token_id`:2150 for open-end generation.


--- german
saved /home/pavlo/lab/TTS/out/clone_german.wav


## Read a whole story in the cloned voice

In [ ]:
PAGES = [
    "It was a sunny day in the vibrant world of Peppa Pig. Peppa had a fantastic idea!",
    "Peppa invited her best friends: George, Suzy Sheep, and Danny Dog.",
]

full = []
for i, page in enumerate(PAGES, 1):
    if SEED is not None:
        torch.manual_seed(SEED); np.random.seed(SEED)
    t0 = time.time()
    wavs, sr = model.generate_voice_clone(text=" ".join(page.split()), ref_audio=REF_WAV,
                                          ref_text=REF_TEXT, language=LANGUAGE,
                                          x_vector_only_mode=X_VECTOR_ONLY)
    a = normalise(np.asarray(wavs[0], dtype=np.float32), sr)
    sf.write(f"{OUT}/clone_page_{i:02d}.wav", a, sr, subtype="PCM_16")
    full.append(a); full.append(np.zeros(int(sr*0.4), dtype=np.float32))
    print(f"page {i:02d}  {len(a)/sr:5.2f}s  gen {time.time()-t0:5.2f}s")

story = np.concatenate(full)
sf.write(f"{OUT}/clone_story_full.wav", story, sr, subtype="PCM_16")
print(f"total {len(story)/sr:.1f}s")
display(Audio(story, rate=sr))

## Free the GPU

In [8]:
try:
    del model
except NameError:
    pass
torch.cuda.empty_cache()
print(gpu_free())

0, NVIDIA GeForce RTX 4090, 5841 MiB
1, NVIDIA GeForce RTX 4060, 7722 MiB
